# Importing Required libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import pickle
import math

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay , accuracy_score

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

from wordcloud import WordCloud
from urllib.parse import urlparse

: 

# Loading Dataset 

In [ ]:


df = pd.read_csv('phish_dataset.csv')
df.head()

# ---  Distribution of URL Types ---

In [ ]:
# ---  Distribution of URL Types ---
plt.figure(figsize=(6, 4))
sns.countplot(x='type', data=df)
plt.title('Distribution of URL Types')
plt.xlabel('Type')
plt.ylabel('Count')
plt.show()

# Generating WordClouds for Each URL Category

In [ ]:
df_phish = df[df.type=='phishing']
df_legi = df[df.type=='legitimate']

In [ ]:
phish_url = " ".join(i for i in df_phish.url)
wordcloud = WordCloud(width=1600, height=800,colormap='Paired').generate(phish_url)
plt.figure( figsize=(12,14),facecolor='k')
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.tight_layout(pad=0)
plt.show()

In [ ]:
legi_url = " ".join(i for i in df_legi.url)
wordcloud = WordCloud(width=1600, height=800,colormap='Paired').generate(legi_url)
plt.figure( figsize=(12,14),facecolor='k')
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.tight_layout(pad=0)
plt.show()

# Check for missing values

In [ ]:
# Check for missing values
print(df.isnull().sum())

# Encoding Target Labels

In [ ]:
label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df['type'])

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

In [ ]:
df

# Lexical Feature Extraction

In [ ]:
# Suspicious keywords
suspicious_keywords = [
    'login', 'signin', 'verify', 'update', 'banking', 'account', 'secure',
    'ebay', 'paypal', 'apple', 'amazon', 'dropbox', 'drive', 'onedrive', 
    'office', 'outlook', 'microsoft', 'cloud', 'confirm', 'password', 
    'credential', 'support', 'service', 'security', 'webscr', 'transfer',
    'refund', 'alert', 'bill', 'invoice', 'gift', 'prize', 'lottery'
]

def shannon_entropy(s):
    """Calculate Shannon entropy of string s."""
    if not s:
        return 0
    probs = [float(s.count(c)) / len(s) for c in dict.fromkeys(list(s))]
    return -sum([p * math.log(p, 2) for p in probs])

def safe_urlparse(url):
    """Safely parse URLs, cleaning invalid IPv6 brackets and handling exceptions."""
    if not isinstance(url, str):
        return urlparse('')
    url = url.strip().replace(' ', '')
    # Remove stray IPv6-like brackets
    url = re.sub(r'[\[\]]', '', url)
    try:
        return urlparse(url)
    except Exception:
        return urlparse('')

def extract_features(url):
    features = {}

    parsed = safe_urlparse(url)
    domain = parsed.netloc
    path = parsed.path
    query = parsed.query
    scheme = parsed.scheme.lower()
    full_url = str(url).lower()

    # --- BASIC FEATURES ---
    features['url_length'] = len(url)
    features['domain_length'] = len(domain)
    features['path_length'] = len(path)
    features['query_length'] = len(query)

    # --- CHARACTER COUNT FEATURES ---
    features['num_digits'] = sum(c.isdigit() for c in url)
    features['num_letters'] = sum(c.isalpha() for c in url)
    features['num_special_chars'] = sum(c in string.punctuation for c in url)
    features['num_uppercase'] = sum(c.isupper() for c in url)
    features['num_lowercase'] = sum(c.islower() for c in url)
    features['num_dashes'] = url.count('-')
    features['num_underscores'] = url.count('_')
    features['num_dots'] = url.count('.')
    features['num_slashes'] = url.count('/')
    features['num_question_marks'] = url.count('?')
    features['num_equals'] = url.count('=')
    features['num_at'] = url.count('@')
    features['num_percent'] = url.count('%')
    features['num_ampersand'] = url.count('&')
    features['num_hash'] = url.count('#')

    # --- DOMAIN FEATURES ---
    features['num_subdomains'] = domain.count('.') - 1 if '.' in domain else 0
    features['has_ip'] = int(bool(re.search(r'\d+\.\d+\.\d+\.\d+', domain)))
    features['has_www'] = int('www' in domain)
    features['has_https'] = int('https' in scheme)
    features['starts_with_http'] = int(url.startswith('http'))
    features['ends_with_slash'] = int(url.endswith('/'))
    features['is_common_tld'] = int(domain.split('.')[-1] in ['com', 'org', 'net', 'edu', 'gov'])
    features['tld_length'] = len(domain.split('.')[-1]) if '.' in domain else 0

    # --- KEYWORD / CONTENT FEATURES ---
    features['has_suspicious_word'] = int(any(word in full_url for word in suspicious_keywords))
    features['has_login_word'] = int(any(k in full_url for k in ['login', 'signin']))
    features['has_bank_word'] = int('bank' in full_url)
    features['has_paypal_word'] = int('paypal' in full_url)
    features['has_ebay_word'] = int('ebay' in full_url)
    features['has_password_word'] = int('password' in full_url)

    # --- STRUCTURAL FEATURES ---
    features['num_params'] = url.count('&') + url.count('=')
    features['num_fragments'] = url.count('#')
    features['num_redirects'] = url.count('//') - 1
    features['has_encoded_chars'] = int(bool(re.search(r'%[0-9a-fA-F]{2}', url)))
    features['repeated_chars'] = int(bool(re.search(r'(.)\1{3,}', url)))
    tokens = [tok for tok in re.split('[./?=-]', url) if tok]
    features['shortest_token_len'] = min([len(tok) for tok in tokens] or [0])
    features['longest_token_len'] = max([len(tok) for tok in tokens] or [0])
    features['avg_token_len'] = sum(len(tok) for tok in tokens) / (len(tokens) or 1)

    # --- ENTROPY / COMPLEXITY FEATURES ---
    features['url_entropy'] = shannon_entropy(url)
    features['domain_entropy'] = shannon_entropy(domain)
    features['path_entropy'] = shannon_entropy(path)

    # --- HEURISTIC / BOOLEAN FEATURES ---
    features['contains_mailto'] = int('mailto:' in url)
    features['contains_double_slash_in_path'] = int('//' in path)
    features['has_long_domain'] = int(len(domain) > 30)
    features['has_long_path'] = int(len(path) > 40)
    features['is_short_url'] = int(any(shortener in full_url for shortener in [
        'bit.ly', 'goo.gl', 'tinyurl', 'ow.ly', 't.co', 'is.gd', 'buff.ly', 'adf.ly'
    ]))

    # --- RATIO FEATURES ---
    features['digit_ratio'] = features['num_digits'] / len(url)
    features['special_char_ratio'] = features['num_special_chars'] / len(url)
    features['letter_ratio'] = features['num_letters'] / len(url)
    features['subdomain_ratio'] = features['num_subdomains'] / (features['num_dots'] + 1)

    return pd.Series(features)

# Apply safely to dataframe
features_df = df['url'].apply(extract_features)
df = pd.concat([df, features_df], axis=1)
df.head()

# Feature Correlation Analysis

In [ ]:
# 3️⃣ BAR PLOT: Correlation of each feature with the target
# 1️⃣ Compute correlation matrix (only numeric features)
numeric_df = df.select_dtypes(include='number')
corr = numeric_df.corr()

# 2️⃣ Compute correlation of each feature with the target label
corr_with_target = corr['label_encoded'].drop('label_encoded').sort_values(key=abs, ascending=False)

# 3️⃣ BAR PLOT: Feature correlations with target
plt.figure(figsize=(7, 10))
sns.barplot(
    x=corr_with_target.values,
    y=corr_with_target.index,
    hue=corr_with_target.index,  # fixes seaborn hue deprecation warning
    palette='coolwarm',
    legend=False
)
plt.title('Feature Correlation with Target (label_encoded)', fontsize=16, weight='bold', pad=15)
plt.xlabel('Correlation Coefficient', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.show()

# Model Training , Evaluation and save 

In [ ]:
# Define features and target
X = df.drop(columns=['url', 'type', 'label_encoded'])
y = df['label_encoded']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize models
models = {
    'XGBoost': XGBClassifier(eval_metric='logloss'),
    'RandomForest': RandomForestClassifier(random_state=42)
}

# Dictionary to store model accuracies
accuracies = {}

# Train and evaluate models
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    accuracies[name] = acc
    print(f"Accuracy for {name}: {acc:.4f}")
    print(f"Classification Report for {name}:")
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))
    print("\n")

    filename = f"{name}_model.pkl"
    with open(filename, "wb") as f:
        pickle.dump(model, f)
    print(f"✅ {name} model saved as '{filename}'\n")

print("🎯 All models trained and saved successfully!")

# Confusion Matrix

In [ ]:
for name, model in models.items():
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
    disp.plot(cmap='Blues', values_format='d')
    plt.title(f"{name} - Confusion Matrix")
    plt.show()

# Model Accuracy Comparisonm

In [ ]:
# Plot model accuracy comparison
plt.figure(figsize=(8, 5))
plt.bar(accuracies.keys(), accuracies.values(), color='skyblue')
plt.title('Model Accuracy Comparison')
plt.ylabel('Accuracy')
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.grid(axis='y')
plt.show()

In [ ]:
# -----------------------------
# 1. Load Model & Encoder
# -----------------------------
with open("RandomForest_model.pkl", "rb") as f:
    model = pickle.load(f)

with open("label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

# -----------------------------
# 2. Helper Functions
# -----------------------------
def shannon_entropy(s):
    """Calculate Shannon entropy of a string (measure of randomness)."""
    if not s:
        return 0
    probs = [float(s.count(c)) / len(s) for c in dict.fromkeys(list(s))]
    return -sum([p * math.log(p, 2) for p in probs])

# -----------------------------
# 3. Feature Extraction Function
# -----------------------------
suspicious_keywords = [
    'login', 'signin', 'verify', 'update', 'banking', 'account', 'secure',
    'ebay', 'paypal', 'apple', 'amazon', 'dropbox', 'drive', 'onedrive', 
    'office', 'outlook', 'microsoft', 'cloud', 'confirm', 'password', 
    'credential', 'support', 'service', 'security', 'webscr', 'transfer',
    'refund', 'alert', 'bill', 'invoice', 'gift', 'prize', 'lottery'
]

shorteners = [
    'bit.ly', 'goo.gl', 'tinyurl', 'ow.ly', 't.co', 'is.gd', 'buff.ly', 'adf.ly'
]

def extract_features(url):
    features = {}

    # Basic counts
    features['url_length'] = len(url)
    features['num_digits'] = sum(c.isdigit() for c in url)
    features['num_special_chars'] = sum(c in string.punctuation for c in url)
    features['num_subdomains'] = url.count('.') - 1
    features['has_ip'] = int(bool(re.search(r'\d+\.\d+\.\d+\.\d+', url)))
    features['has_https'] = int('https' in url.lower())
    features['num_params'] = url.count('?')
    features['num_fragments'] = url.count('#')
    features['num_slashes'] = url.count('/')

    # Keyword & content indicators
    features['has_suspicious_words'] = int(any(word in url.lower() for word in suspicious_keywords))
    features['contains_mailto'] = int('mailto:' in url)
    features['contains_double_slash_in_path'] = int('//' in url)

    # TLD-related
    tld = url.split('.')[-1]
    features['tld_length'] = len(tld)
    features['is_common_tld'] = int(tld in ['com', 'org', 'net', 'edu', 'gov'])

    # Encodings and repetitions
    features['has_hex'] = int(bool(re.search(r'%[0-9a-fA-F]{2}', url)))
    features['repeated_chars'] = int(bool(re.search(r'(.)\1{3,}', url)))

    # Ratios
    features['digit_ratio'] = features['num_digits'] / len(url) if len(url) > 0 else 0
    features['special_char_ratio'] = features['num_special_chars'] / len(url) if len(url) > 0 else 0
    features['letter_ratio'] = sum(c.isalpha() for c in url) / len(url) if len(url) > 0 else 0

    # Entropy (randomness)
    features['url_entropy'] = shannon_entropy(url)
    features['domain_entropy'] = shannon_entropy(url.split('/')[2]) if '/' in url else 0
    features['path_entropy'] = shannon_entropy(url.split('/')[-1]) if '/' in url else 0

    # Shortened / suspicious indicators
    features['is_short_url'] = int(any(shortener in url for shortener in shorteners))
    features['has_long_domain'] = int(len(url.split('/')[2]) > 30) if '/' in url else 0
    features['has_long_path'] = int(len(url.split('/')[-1]) > 40) if '/' in url else 0

    # Additional structural ratios
    features['subdomain_ratio'] = features['num_subdomains'] / (url.count('.') + 1)

    return pd.DataFrame([features])

# -----------------------------
# 4. Test URL Prediction
# -----------------------------
test_url = "https://chatgpt.com/c/6909f473-c60c-8324-a8c1-56612e722c58"

# Extract features
X_new = extract_features(test_url)

# Align feature order to model training order (optional safety)
X_new = X_new.reindex(columns=model.feature_names_in_, fill_value=0)

# Predict
y_pred = model.predict(X_new)[0]
label = label_encoder.inverse_transform([y_pred])[0]

print(f"🔍 URL: {test_url}")
print(f"🧾 Predicted Class: {label}")



In [ ]:
# 1. leakage_checks.py
import numpy as np
import pandas as pd

# assume df already loaded and label encoded as in your notebook
# df = pd.read_csv('phishing_dataset.csv')  # if needed
# label encoding already done in your notebook: 'label_encoded'

# 1.1 Confirm label exists and type
print("label_encoded dtype:", df['label_encoded'].dtype)
print("Label counts:\n", df['label_encoded'].value_counts())

# 1.2 Make sure we only consider numeric features (exclude url, type, etc.)
numeric_df = df.select_dtypes(include=['number']).copy()

# 1.3 Are there any columns equal to label exactly?
equal_to_label = [col for col in numeric_df.columns if (numeric_df[col].equals(numeric_df['label_encoded']))]
print("Columns exactly equal to label:", equal_to_label)

# 1.4 Perfect correlations with label (corr == 1.0 or -1.0)
corr_with_label = numeric_df.corr()['label_encoded'].drop('label_encoded')
perfect_corr = corr_with_label[ corr_with_label.abs() >= 0.9999 ]  # tolerance for floating point
print("\nPerfect or near-perfect correlations with label:\n", perfect_corr)

# 1.5 Any single feature that is constant per class? (value uniquely identifies class)
problematic = []
for col in corr_with_label.index:
    pivot = pd.crosstab(df[col], df['label_encoded'])
    # if for every unique value of col, it maps to a single label: possible leakage
    if (pivot.apply(lambda row: (row>0).sum(), axis=1) == 1).all():
        problematic.append(col)
print("\nColumns whose distinct values map 1-to-1 to classes (possible leakage):", problematic)

# 1.6 Duplicated rows
dups = df.duplicated().sum()
print("\nNumber of duplicated rows in dataset:", dups)

# 1.7 Print top correlations with label to inspect
print("\nTop 10 absolute correlations with label:")
print(corr_with_label.abs().sort_values(ascending=False).head(10))
